# A/B TEST: 10% OFF SECOND PURCHASE COUPON

## Business Problem
Olist has a very high one-time buyer concentration. Based on customer aggregation:
Total customers: 93,358
Eligible first-time buyers: 90,557 (~97%)

These first-time buyers represent the primary opportunity for improving repeat purchase behavior through targeted interventions. 


## Hypothesis
Offering a 10% discount coupon for the second purchase (sent 24 hours after first order, valid for 30 days) will significantly increase repeat purchase rate among first-time buyers.



## Experiment Design

### CONTROL GROUP
- No coupon sent after first purchase
- Represents baseline customer behavior
- Expected repeat rate: 3% (based on historical data)

### TREATMENT GROUP  
- Receives 10% off coupon via email 24 hours after first purchase
- Coupon valid for 30 days
- One-time use per customer
- Expected repeat rate (treatment): 4.2%
- This represents an assumed relative lift of ~40% over the 3% baseline and is used only for simulation purposes.

Note: Actual observed results may differ from this design assumption.

# Simulation Note:
- This portfolio project uses simulated experiment outcomes to demonstrate proper A/B testing methodology and statistical evaluation. In a production environment, these values would be derived from live experiment data collected during the test period.

**In a real production scenario, these values would come from live experiment data collected over the test period.**

### WHY THIS DESIGN?

**Why 10% discount?**
- Large enough to create urgency (psychological threshold)
- `Small enough to maintain healthy margins (avg order R$140, discount = R$14)`
- Industry standard for retention campaigns

**Why 24-hour delay?**
- Allows time for first package to ship (builds anticipation)
- Avoids overwhelming customer immediately after purchase
- Optimal timing for retention email open rates

**Why 30-day validity?**
- Long enough to not feel rushed (reduces pressure)
- Short enough to create urgency (avoids infinite delay)
- Matches typical repurchase cycle for marketplace items

**Why 50/50 split?**
- Equal sample sizes maximize statistical power
- Standard A/B testing practice
- Ensures no segment is under-served during test

### Sample Size & Power
- Eligible population: 90,557 one-time buyers
- Control: 45,200 customers
- Treatment: 45,357 customers
- With this sample size, we can detect even small lifts (1-2%) with high confidence
- Expected power: >95% to detect 40% relative lift

### Primary Metric
Repeat purchase rate within 30 days of first order

## Decision Rule
The experiment will be considered successful if the treatment group shows a statistically significant improvement (p < 0.05) in repeat purchase rate without negatively impacting revenue per customer.

### Secondary Metrics
- Time to second purchase (days)
- Average revenue per customer (first + second order)
- Coupon redemption rate (treatment group only)

In [1]:
# Install if needed (run once)
!pip install sqlalchemy pymysql scipy

In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from scipy import stats

In [3]:
pd.set_option('display.max_columns', None)

In [4]:
import os
from dotenv import load_dotenv
from urllib.parse import quote_plus

In [5]:
load_dotenv()

username = os.getenv("DB_USERNAME")
password = quote_plus(os.getenv("DB_PASSWORD"))
host = os.getenv("DB_HOST")
database = os.getenv("DB_DATABASE")

In [6]:
engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:3306/{database}",
    pool_pre_ping=True
)

In [7]:
query = """
SELECT
    customer_unique_id,
    order_id,
    order_purchase_timestamp,
    order_revenue
FROM fact_orders
WHERE order_status = 'DELIVERED'
"""

In [8]:
orders = pd.read_sql(query, engine)

In [9]:
print(orders.shape)
orders.head()

(96478, 4)


,customer_unique_id,order_id,order_purchase_timestamp,order_revenue
0,871766c5855e863f6eccc05f988b23cb,00010242fe8c5a6d1ba2dd792cb16214,2017-09-13 08:59:02,58.90
1,eb28e67c4c0b83846050ddfb8a35d051,00018f77f2f0320c557190d7a144bdd3,2017-04-26 10:53:06,239.90
2,3818d81c6709e39d06b2738a8d3a2474,000229ec398224ef6ca0657da4fc703e,2018-01-14 14:33:31,199.00
3,af861d436cfc08b2c2ddefd0ba074622,00024acbcdf0a6daa1e931b038114c75,2018-08-08 10:00:35,12.99
4,64b576fb70d441e8f1b2d7d446e483c5,00042b26cf59d7ce69dfabb4e55b4fd9,2017-02-04 13:57:51,199.90


In [10]:
# =============================
# Build customer-level summary
# =============================

customer_base = (
    orders.groupby("customer_unique_id", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_revenue=("order_revenue", "sum"),
        first_order_date=("order_purchase_timestamp", "min"),
        last_order_date=("order_purchase_timestamp", "max"),
    )
)

In [11]:
customer_base.head()
print("Total customers:", customer_base.shape[0])

Total customers: 93358


In [12]:
# =============================
# Eligible population
# =============================

eligible_customers = customer_base[
    customer_base["total_orders"] == 1
].copy()

In [13]:
print("Eligible customers:", eligible_customers.shape[0])
eligible_customers.head()

Eligible customers: 90557


,customer_unique_id,total_orders,total_revenue,first_order_date,last_order_date
0,0000366f3b9a7992bf8c76cfdf3221e2,1,129.90,2018-05-10 10:56:27,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,18.90,2018-05-07 11:11:27,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,1,69.00,2017-03-10 21:05:03,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,1,25.99,2017-10-12 20:29:41,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,1,180.00,2017-11-14 19:45:42,2017-11-14 19:45:42


In [14]:
eligible_customers["total_orders"].value_counts()

total_orders
1    90557
Name: count, dtype: int64

In [15]:
# =============================
# Random A/B assignment
# =============================

np.random.seed(42)  # ensures reproducibility

eligible_customers["group"] = np.random.choice(
    ["Control", "Treatment"],
    size=len(eligible_customers),
    p=[0.5, 0.5]
)

eligible_customers["group"].value_counts()

group
Treatment    45357
Control      45200
Name: count, dtype: int64

In [16]:
# =============================
# Define probabilities
# =============================

control_prob = 0.03      # based on observed repeat rate (~3%)
treatment_prob = 0.042   # assume ~40% relative uplift

print("Control prob:", control_prob)
print("Treatment prob:", treatment_prob)

Control prob: 0.03
Treatment prob: 0.042


In [17]:
# =============================
# Simulate repeat behavior
# =============================

# generate random numbers
eligible_customers["rand"] = np.random.rand(len(eligible_customers))

# assign probability by group
eligible_customers["repeat_flag"] = np.where(
    eligible_customers["group"] == "Control",
    eligible_customers["rand"] < control_prob,
    eligible_customers["rand"] < treatment_prob
)

# convert to int
eligible_customers["repeat_flag"] = eligible_customers["repeat_flag"].astype(int)

eligible_customers.head()

,customer_unique_id,total_orders,total_revenue,first_order_date,last_order_date,group,rand,repeat_flag
0,0000366f3b9a7992bf8c76cfdf3221e2,1,129.90,2018-05-10 10:56:27,2018-05-10 10:56:27,Control,0.980462,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,18.90,2018-05-07 11:11:27,2018-05-07 11:11:27,Treatment,0.274168,0
2,0000f46a3911fa3c0805444483337064,1,69.00,2017-03-10 21:05:03,2017-03-10 21:05:03,Treatment,0.825563,0
3,0000f6ccb0745a6a4b88665a16c9f078,1,25.99,2017-10-12 20:29:41,2017-10-12 20:29:41,Treatment,0.502230,0
4,0004aac84e0df4da2b147fca70cf8255,1,180.00,2017-11-14 19:45:42,2017-11-14 19:45:42,Control,0.799221,0


In [18]:
# =============================
# Experiment results
# =============================

ab_results = (
    eligible_customers.groupby("group")
    .agg(
        customers=("customer_unique_id", "count"),
        repeat_rate=("repeat_flag", "mean")
    )
)

ab_results["repeat_rate_pct"] = ab_results["repeat_rate"] * 100

In [19]:
ab_results_display = ab_results.copy()

ab_results_display["repeat_rate_pct"] = (
    ab_results_display["repeat_rate_pct"].round(2)
)

ab_results_display

,customers,repeat_rate,repeat_rate_pct
group,,,
Control,45200,0.029513,2.95
Treatment,45357,0.040854,4.09


In [20]:
# =============================
# Prepare inputs for z-test
# =============================

control = eligible_customers[eligible_customers["group"] == "Control"]
treatment = eligible_customers[eligible_customers["group"] == "Treatment"]

# successes (repeat customers)
success_control = control["repeat_flag"].sum()
success_treatment = treatment["repeat_flag"].sum()

# totals
n_control = len(control)
n_treatment = len(treatment)

print(success_control, success_treatment)
print(n_control, n_treatment)

1334 1853
45200 45357


# Two-Proportion Z-Test for Repeat Purchase Rate

**Null Hypothesis (H0)**: Repeat purchase rate is the same for control and treatment groups

**Alternative Hypothesis (H1)**: Repeat purchase rate differs between the groups

**Why two-proportion z-test?**
- Outcome is binary (repeat vs no repeat)
- We are comparing proportions between two independent groups
- Appropriate for large-sample A/B testing scenarios

**Interpretation:**
- p-value < 0.05 → statistically significant difference (reject H0)
- p-value ≥ 0.05 → no statistically significant difference detected

In [21]:
# =============================
# Two-proportion z-test
# =============================

from statsmodels.stats.proportion import proportions_ztest

successes = [success_control, success_treatment]
samples = [n_control, n_treatment]

z_stat, p_value = proportions_ztest(successes, samples)

print("Z-stat:", round(z_stat, 4))

Z-stat: -9.2599


In [22]:
alpha = 0.05

if p_value < alpha:
    print("Result: Statistically significant at 5% level")
else:
    print("Result: Not statistically significant")

Result: Statistically significant at 5% level


In [23]:
control_rate = ab_results.loc["Control", "repeat_rate"]
treatment_rate = ab_results.loc["Treatment", "repeat_rate"]

absolute_uplift = treatment_rate - control_rate
relative_uplift = absolute_uplift / control_rate * 100

print(f"Absolute uplift: {absolute_uplift*100:.2f}%")
print(f"Relative uplift: {relative_uplift:.2f}%")

Absolute uplift: 1.13%
Relative uplift: 38.42%


## Business Impact (traceable)

The figures shown on the dashboard are **computed in the cell below** from the experiment outputs (`control_rate`, `treatment_rate`, `n_treatment`) plus a single **stated assumption** — second-order value = R\$137 (the measured AOV) — because the simulation generated repeat *behaviour* (yes/no), not revenue.

**Incrementality:** the 10% coupon cost is paid by *every* treatment repeater, but only the orders *above the control baseline* are net-new revenue. Customers who would have returned anyway are now being discounted — the calculation below accounts for this, so the ROI is conservative and reproducible rather than asserted.

In [24]:
# ============================================================
# BUSINESS IMPACT - traceable ROI
# Revenue is an EXPLICIT assumption: the simulation generated repeat
# behaviour (yes/no), not revenue, so a second-order value must be assumed.
# Everything else is derived from earlier cells (control_rate,
# treatment_rate, success_treatment, n_treatment).
# ============================================================

# --- Stated assumptions ---
AOV        = 137.0   # R$, = measured average order value (dashboard KPI ~R$136.7)
COUPON_PCT = 0.10    # 10% off the second purchase
REDEMPTION = 1.00    # assume every 2nd-purchase customer redeems the coupon

abs_lift = treatment_rate - control_rate   # causal lift (from earlier cell)

# ---- Per 10,000 new first-time buyers / month (the dashboard framing) ----
cohort             = 10_000
ctrl_repeaters     = control_rate   * cohort
trt_repeaters      = treatment_rate * cohort
incremental        = trt_repeaters - ctrl_repeaters                  # net-new (causal) repeaters
gross_incr_revenue = incremental * AOV
coupon_cost        = trt_repeaters * AOV * COUPON_PCT * REDEMPTION   # paid by ALL redeemers, not just incremental
net_gain           = gross_incr_revenue - coupon_cost
roi                = net_gain / coupon_cost

impact = pd.DataFrame({
    "Metric": ["Repeaters (control)", "Repeaters (treatment)", "Incremental repeaters",
               "Gross incremental revenue", "Coupon cost", "Net gain (monthly)",
               "Annualized net gain", "ROI"],
    "Value":  [f"{ctrl_repeaters:.0f}", f"{trt_repeaters:.0f}", f"+{incremental:.0f}",
               f"R$ {gross_incr_revenue:,.0f}", f"R$ {coupon_cost:,.0f}", f"R$ {net_gain:,.0f}",
               f"R$ {net_gain*12:,.0f}", f"{roi:.0%}"]
})

print(f"Per 10,000 new first-time buyers / month  (AOV assumption = R${AOV:.0f})")
print(impact.to_string(index=False))

# NOTE: ROI is robust to the AOV assumption (revenue and cost both scale with AOV);
# only the absolute R$ magnitudes change. The coupon discounts EVERY treatment
# repeater, but only the orders above the control baseline are net-new revenue.

Per 10,000 new first-time buyers / month  (AOV assumption = R$137)
                   Metric      Value
      Repeaters (control)        295
    Repeaters (treatment)        409
    Incremental repeaters       +113
Gross incremental revenue  R$ 15,536
              Coupon cost   R$ 5,597
       Net gain (monthly)   R$ 9,939
      Annualized net gain R$ 119,273
                      ROI       178%


## Test Sensitivity: Power & Minimum Detectable Effect (MDE)

The design section earlier *asserted* ">95% power." The cell below **computes** it with `statsmodels` (two-proportion, two-sided, α = 0.05) instead:

- **Design power** — probability the test detects the pre-registered 3.0% → 4.2% lift at the realized sample size (~45K/group).
- **MDE** — the smallest lift the test could reliably catch at 80% power, given the sample size and the observed ~2.95% baseline. The observed lift is then compared against the MDE to confirm the test was adequately sensitive.

In [25]:
# ============================================================
# TEST SENSITIVITY: statistical power & minimum detectable effect (MDE)
# Computed with statsmodels (two-proportion, two-sided, alpha = 0.05),
# replacing the asserted ">95% power" with an actual calculation.
# Uses n_control, n_treatment, control_rate, treatment_rate from earlier cells.
# ============================================================
import numpy as np
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

alpha        = 0.05
target_power = 0.80
ratio        = n_treatment / n_control          # group-size ratio (~1.0)
analysis     = NormalIndPower()

# (1) DESIGN POWER -- ability to detect the pre-registered assumption (3.0% -> 4.2%)
p1_design, p2_design = 0.030, 0.042
h_design     = abs(proportion_effectsize(p1_design, p2_design))   # Cohen's h
power_design = analysis.solve_power(effect_size=h_design, nobs1=n_control,
                                    alpha=alpha, ratio=ratio, alternative="two-sided")
print(f"Design power to detect {p1_design:.1%} -> {p2_design:.1%} "
      f"(n1={n_control:,}): {power_design:.1%}")

# (2) MDE -- smallest lift detectable at 80% power, given realized sample & baseline
p1     = control_rate                                              # observed baseline (~2.95%)
h_mde  = analysis.solve_power(effect_size=None, nobs1=n_control, alpha=alpha,
                              power=target_power, ratio=ratio, alternative="two-sided")
phi1   = 2 * np.arcsin(np.sqrt(p1))
p2_mde = np.sin((phi1 + h_mde) / 2) ** 2          # invert Cohen's h back to a proportion (uplift)
mde_abs = p2_mde - p1
mde_rel = mde_abs / p1
print(f"MDE @ {target_power:.0%} power (baseline {p1:.2%}): "
      f"+{mde_abs*100:.2f} pp  ->  {p2_mde:.2%}  ({mde_rel:+.0%} relative)")

obs_lift = treatment_rate - control_rate
print(f"Observed lift +{obs_lift*100:.2f} pp is "
      f"{'ABOVE' if obs_lift > mde_abs else 'BELOW'} the MDE -- "
      f"the test was sensitive enough to detect the effect.")

Design power to detect 3.0% -> 4.2% (n1=45,200): 100.0%
MDE @ 80% power (baseline 2.95%): +0.32 pp  ->  3.27%  (+11% relative)
Observed lift +1.13 pp is ABOVE the MDE -- the test was sensitive enough to detect the effect.


In [26]:
# Write A/B test summary to MySQL for Power BI

# Reset index so 'group' becomes a column (not index)
ab_summary = ab_results_display.reset_index()

# Push to MySQL
ab_summary.to_sql('ab_test_summary', con=engine, if_exists='replace', index=False)

print("Pushed ab_test_summary to MySQL")
print(ab_summary)


Pushed ab_test_summary to MySQL
       group  customers  repeat_rate  repeat_rate_pct
0    Control      45200     0.029513             2.95
1  Treatment      45357     0.040854             4.09


## Findings

### Primary Metric: Repeat Purchase Rate
- Control: 2.95%  
- Treatment: 4.09%  
- Absolute lift: +1.13 percentage points  
- Relative lift: +38.4%  
- Statistical test: Two-proportion z-test  
- p-value: < 0.001  

**Conclusion:** The coupon intervention produces a statistically significant increase in repeat purchase rate.

**Note:** The observed treatment repeat rate (4.09%) is slightly below the assumed 4.2% used during experiment design. However, the uplift remains statistically significant and economically positive, validating the effectiveness of the coupon strategy.

---

### Secondary Metric: Revenue per Customer
Revenue impact is directionally estimated based on simulated repeat behavior.

Treatment group shows higher expected revenue per customer driven by increased repeat probability.

A formal revenue t-test should be conducted on observed customer-level revenue in a production experiment.

Conclusion:
The primary validated impact of the experiment is the statistically significant lift in repeat purchase rate. Revenue uplift should be validated using real post-treatment transaction data.
---

## Business Impact

Assuming 10,000 new first-time customers per month:

Incremental repeat buyers ≈ +113 customers
(based on +1.13 pp absolute lift)

Estimated incremental revenue ≈ R$11K per month
(using average order value as proxy)

Annualized impact ≈ R$130K+

Note: These projections are directional and should be validated with live experiment data and full discount cost accounting..

---
### Power and Sensitivity

The experiment was conducted on a large sample of first-time buyers, providing sufficient statistical power to detect meaningful changes in repeat purchase behavior.

With control conversion at ~2.95% and an observed absolute lift of +1.13 percentage points, the effect size is materially significant for an ecommerce retention intervention.

**Interpretation:**  
The observed improvement is unlikely to be driven by random variation and is large enough to be operationally meaningful.

---

### Minimum Detectable Effect (MDE) Perspective

From a practical standpoint, the experiment successfully detects an effect above a meaningful business threshold.

- Observed absolute lift: **+1.13 pp**  
- Typical ecommerce actionable range: **0.5–1.0 pp**

**Conclusion:**  
The measured uplift exceeds common minimum detectable effect thresholds, supporting rollout consideration.

---

### Guardrail Metrics to Monitor

While repeat conversion improved, a production rollout should monitor key guardrail metrics to ensure no adverse impact on unit economics.

Recommended guardrails:

- Average Order Value (AOV)  
- Refund and cancellation rate  
- Discount cost per incremental order  
- Customer support ticket volume  
- Contribution margin after discount  

**Rationale:**  
Discounts can increase repeat purchases but may also reduce margins, so overall profitability should be validated after rollout.

---

### Incrementality and Profitability Considerations

The experiment demonstrates statistically significant improvement in repeat purchase behavior. However, full business validation requires confirming economic incrementality.

Before full rollout, the business should evaluate:

- Compare the extra revenue generated with the total discount cost  
- Check whether the coupon is driving new repeat purchases or just discounting customers who would have returned anyway  
- Monitor whether customers continue purchasing after the second order  
- Evaluate long-term customer lifetime value (LTV) by cohort  

## Final Recommendation

Based on the A/B test results:

- The coupon drives a statistically significant lift in repeat purchases  
- Revenue per customer improves even after accounting for discount cost  
- The intervention shows strong ROI potential at scale  

**Recommendation:** Proceed with a phased rollout of the 10% second-purchase coupon while continuing to monitor real-world performance metrics.

---

## Important Note

This analysis is based on simulated experiment data for portfolio demonstration. In a production environment, results should be validated using live A/B test outcomes before full-scale deployment.